---
<font color='Blue' size="4">
F37.206 컴퓨팅 탐색: 실생활에서 활용하기(Exploring Computing: Applications in Everyday Life)</font>

---


# Chapter 7. 게임 고도화하기

<div style="background-color: #f5fff5; padding: 10px; border-radius: 5px; color: #000000;">

<font size=5> <strong> <mark style="background-color: #f5fff5;">✅ 학습목표와 기대효과 </mark></strong></font>

<mark style="background-color: #f5fff5;">
🔹 학습목표<br>
  <ul><li> Pygame을 활용하여 사운드와 배경음악을 효과적으로 적용하는 방법을 익힌다.</li>
  <li> 게임의 흐름에 맞는 논리 구조와 규칙을 구현해본다.</li>
  <li> 난이도 조절을 위한 레벨 구성과 장면 전환 기능을 구현해본다.</li></ul>    
🔹 기대효과<br>
  <ul><li> 게임의 몰입도를 높이는 요소들을 직접 구현하며 프로그래밍 능력을 향상시킬 수 있다.</li>
  <li> 다양한 게임 장르에 활용 가능한 기초 제작 기술을 습득하게 된다. </li></ul> 
</div>

## <div style="background-color:rgba(208, 205, 208, 1); padding: 10px; border-radius: 5px;"><mark style="background-color: rgba(208, 205, 208, 1);">**템플릿 소스 코드**</mark></div>

<div align="center"><img src="https://haesunbyun.github.io/common/images/comps/image20.png?v=1234" width=400>  </div>

- 아래 코드는 지난주까지 배운 소스코드이다. 

In [ ]:
import pygame
import random

pygame.init()

screen = pygame.display.set_mode((800, 600))
pygame.display.set_caption("풍선을 잡아라!")
clock = pygame.time.Clock()


class Player(pygame.sprite.Sprite):
    def __init__(self):
        super().__init__()
        self.image = pygame.image.load('./images/player.jpg').convert_alpha()
        self.rect = self.image.get_rect()
        self.rect.center = (400, 300)
        self.velocity = pygame.math.Vector2(0, 0)

    def update(self):
        self.velocity.x = 0  # 매 프레임 좌우 속도 초기화
        self.velocity.y += 0.5 #

        keys = pygame.key.get_pressed()
        if keys[pygame.K_LEFT]:
            self.velocity.x = -5  
        if keys[pygame.K_RIGHT]:
            self.velocity.x = 5
        if keys[pygame.K_UP] and self.rect.bottom >= 600:
            self.velocity.y = -10  # Jump

        self.rect.x += self.velocity.x
        self.rect.y += self.velocity.y

        if self.rect.bottom >= 600:
            self.rect.bottom = 600
            self.velocity.y = 0

        # 화면을 벋어나는 것을 방지    
        if self.rect.left <= 0:
            self.rect.left = 0
        if self.rect.right >= 800:
            self.rect.right = 800  

################################################## 풍선이 아래로 내려오는 상황
class Enemy(pygame.sprite.Sprite):
    def __init__(self):
        super().__init__()
        self.image = pygame.image.load('./images/balloon.png').convert_alpha()
        self.image = pygame.transform.scale(self.image, (60, 80))
        self.rect = self.image.get_rect()
        self.rect.center = (random.randint(50, 750), random.randint(-100, -40)) 
        self.velocity = random.uniform(1.0, 2.5)  # 풍선 낙하 속도
        self.mask = pygame.mask.from_surface(self.image)

    def update(self):
        self.rect.y += self.velocity
        if self.rect.top > 600:  # 화면 아래로 나가면 제거
            self.kill()

all_sprites = pygame.sprite.Group()
player = Player()
all_sprites.add(player)

balloon_group = pygame.sprite.Group() 

# 처음에 3개 풍선
for _ in range(3):
    balloon_group.add(Enemy())


running = True
while running:
    screen.fill((255, 255, 255))    
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    while len(balloon_group) < 3:
        balloon_group.add(Enemy())   

    all_sprites.update()  # Update all sprites in the group
    all_sprites.draw(screen)  # Draw all sprites in the group

    balloon_group.update()        
    balloon_group.draw(screen)

    pygame.display.flip()
    clock.tick(60)    

pygame.quit()

<div style="background-color:rgba(247, 239, 246, 1); padding: 10px; border-radius: 5px;">
<mark style="background-color: rgba(247, 239, 246, 1);">
📢 안내<br> 
- 템플릿 소스 코드를 확장하면서 위아래로 자주 이동해야 하므로, 템플릿 코드를 새 파일에 복사해 두고 원본 파일과 번갈아 보며 작업해주세요.<br>
- 메뉴에서 [파일(File)] → [새 파일(New File)] → 파일이름: **Ex_ch7_source.ipynb** -> 저장위치 지정후 엔터
</mark></div>

## <div style="background-color:rgba(208, 205, 208, 1); padding: 10px; border-radius: 5px;"><mark style="background-color: rgba(208, 205, 208, 1);">**사운드와 음악 추가**</mark></div>

- 사운드와 음악은 게임 개발에서 중요한 요소로, 게임의 깊이와 몰입감을 더해준다. 
- Pygame은 사운드와 음악을 로드하고, 재생하며, 관리하는 간단하고 효과적인 방법을 제공한다.
- 사운드 효과는 게임 내 특정 행동이나 이벤트(예: 점프, 총알 발사, 객체와 충돌 등)를 강조하기 위해 사용되는 짧은 오디오 클립이다. 
- Pygame은 `pygame.mixer` 모듈을 사용하여 사운드 효과를 로드하고 재생할 수 있다.

### ✅ 사운드 효과 넣기
- 믹서 초기화: 사운드를 로드하고 재생하기 전에, 믹서를 초기화해야 한다. 

In [ ]:
import pygame

pygame.mixer.init() 

- 사운드 효과 로드하기
    - 사운드 효과를 로드하려면 `pygame.mixer.Sound()` 함수를 사용한다.
    - 음악파일을 sound 폴더 하나 만들어서 그 안에 넣어 놓도록 한다. 
    - 아래 링크를 클릭하여 사운드 위젯이 나오면 마우스 오른쪽 버튼 눌러 다운로드 한후, sound 폴더에 저장한다. 
    - https://haesunbyun.github.io/common/images/jump.mp3 (무료배포 가능한 음악임 https://artlist.io)


In [ ]:
jump_sound = pygame.mixer.Sound('./sound/jump.mp3')

- 사운드 효과 재생하기
    - 사운드 효과를 로드한 후, `Sound` 객체의 `play()` 메서드를 사용하여 해당 사운드를 재생할 수 있다.

In [ ]:
running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        elif event.type == pygame.KEYDOWN:
            if event.key == pygame.K_UP:
                jump_sound.play()

### 🚀**해보기 1: 사운드 효과 넣기**
<div style="background-color:rgba(247, 239, 246, 1); padding: 10px; border-radius: 5px;">
<mark style="background-color: rgba(247, 239, 246, 1);">
Ex_ch7_source.ipynb에 사운드 효과 넣기 코드를 추가하세요.
</mark></div>

---

### ✅ 배경 음악 넣기

- 배경 음악 로드하기
    - 배경 음악은 지속적으로 재생되며 게임의 전반적인 분위기를 설정하는 역할을 한다. 
    - 사운드 효과와 달리 배경 음악은 보통 게임 플레이 중에 반복되는 긴 오디오 트랙이다.
    - 배경 음악을 로드하려면 `pygame.mixer.music.load()` 함수를 사용한다.
    - 아래 링크를 클릭하여 사운드 위젯이 나오면 마우스 오른쪽 버튼 눌러 다운로드 한후, sound 폴더에 저장한다. 
    - https://haesunbyun.github.io/common/images/background.mp3 (무료배포 가능한 음악임 https://artlist.io)
    - 이 배경 음악이 맘에 들지 않으면 다른 음악을 사용해도 됨

In [ ]:
# 배경음악 로드하기
pygame.mixer.music.load('./sound/background.mp3')

- 배경 음악 재생하기
    - 배경 음악을 재생하려면 `pygame.mixer.music.play()` 함수를 사용한다. 
    - 이 함수의 옵션으로는 `loops`와 `start`가 있다.
    - **loops**: 음악을 반복할 횟수. `-1`로 설정하면 음악이 무한 반복된다.
    - **start**: 음악 파일에서 재생을 시작할 위치 (초 단위)

In [ ]:
# 배경음악 무한 반복 재생하기
pygame.mixer.music.play(loops=-1)

- 음악 정지 및 일시 정지
    - 배경 음악 재생을 제어하려면 `stop()`과 `pause()` 메서드를 사용할 수 있습니다.
    - **stop()**: 음악을 즉시 정지
    - **pause()**: 음악을 일시 정지 
    - **unpause()**: 음악을 다시 재개
    - 이 기능들을 pause와 unpause는 s키로 토글기능을 갖고 하고, stop은 x 키와 연동시켜보자. 

In [ ]:
# 음악 일시정지 상태 변수
paused = False

if event.key == pygame.K_s:  # S 키로 토글
    if paused:
        pygame.mixer.music.unpause()
        paused = False
    else:
        pygame.mixer.music.pause()
        paused = True
elif event.key == pygame.K_x:  # X 키로 완전 정지
    pygame.mixer.music.stop()

- 볼륨 설정
    - 배경 음악의 볼륨은 `set_volume()` 메서드를 사용하여 조정할 수 있다. 
    - 볼륨 레벨은 0.0에서 1.0 사이의 값으로 설정할 수 있다.

In [ ]:
pygame.mixer.music.set_volume(0.5)  # 50% 볼륨

- pygame에서 윈도우를 닫아도 배경음악이 멈추지 않고 계속 재생될 때에는 음악이 백그라운드에서 계속 실행되어 발생한다. 
- 그럴 경우 메뉴에서 Kernel → Restart

### 🚀**해보기 2: 배경 음악 넣기**
<div style="background-color:rgba(247, 239, 246, 1); padding: 10px; border-radius: 5px;">
<mark style="background-color: rgba(247, 239, 246, 1);">
Ex_ch7_source.ipynb에 배경 음악 넣기 코드를 추가하세요.
</mark></div>

---

## <div style="background-color:rgba(208, 205, 208, 1); padding: 10px; border-radius: 5px;"><mark style="background-color: rgba(208, 205, 208, 1);">**게임 논리 및 규칙**</mark></div>

- 게임 논리와 규칙을 구현하는 것은 게임 개발의 핵심이다. 
- 게임이 어떻게 동작하는지, 플레이어가 게임 세계와 어떻게 상호작용하는지, 게임이 어떻게 진행되는지를 결정해야 한다. 
- 게임 논리와 규칙에는 캐릭터의 이동, 적의 행동, 승리와 패배 조건 등 게임의 모든 측면이 포함되어야 한다.
- 이전에 우리는 캐릭터와 풍선의 이동을 구현해 보았다. 
- 이번에는 캐릭터와 풍선의 충돌감지를 구현해보자.

### ✅ 충돌 감지

- 충돌 감지는 게임 플레이에서 매우 중요하다. 예를 들어, 플레이어 스프라이트가 적, 벽 또는 다른 객체와 충돌할 때 이를 감지하는 기능이 필요하다. Pygame은 여러 가지 충돌 감지 방법을 제공한다.

- 충돌 감지 함수는 다음과 같다. 

    <font size=2>

    | 함수 이름                          | 설명 |
    |-----------------------------------|------|
    | `pygame.sprite.collide_rect(a, b)` | 두 스프라이트의 `rect` 사각형이 겹치는지 검사 (단순 박스 충돌) |
    | `pygame.sprite.collide_circle(a, b)` | 두 스프라이트의 중심 좌표 거리로 충돌 검사 (`radius` 속성이 필요함) |
    | `pygame.sprite.collide_mask(a, b)` | 마스크 기반 픽셀 단위 충돌 검사 (더 정밀한 충돌을 감지할 수 있게 해줌) |
    | `pygame.sprite.spritecollide(sprite, group, dokill)` | 하나의 스프라이트가 그룹 내 어떤 스프라이트와 충돌하는지 검사 |
    | `pygame.sprite.spritecollideany(sprite, group)` | 그룹 내 스프라이트 중 하나라도 충돌하면 True 반환 |
    | `pygame.sprite.groupcollide(group1, group2, dokill1, dokill2)` | 두 그룹 간 충돌 검사 및 충돌한 항목 반환 |

    </font>

In [ ]:
# 단순 박스 충돌
for balloon in balloon_group:
    if pygame.sprite.collide_rect(player, balloon):
        print("충돌!")

- 마스크(Mask): 마스크는 스프라이트 이미지에서 픽셀의 투명도를 나타내는 1비트 이미지이다. 
- 마스크에서 픽셀이 검은색이면 해당 위치가 투명하다는 의미하고, 흰색이면 불투명한 픽셀을 의미한다. 
- 이를 통해 투명 영역을 제외한 실제 이미지의 충돌만 감지할 수 있다.

In [ ]:
# player와 balloon 모두 마스크 설정
self.mask = pygame.mask.from_surface(self.image)  # 마스크 생성

# 충돌 검사
hit = pygame.sprite.spritecollide(player, balloon_group, True, pygame.sprite.collide_mask)
if hit:
    for _ in range(len(hit)):
        balloon_group.add(Enemy())

- 충돌 검사를 위해 player와 balloon_group의 마스크가 겹치는지 검사한다. 
- 충돌하였다면 풍선이 제거가 되어야 하는데 세 번째 인자 True가 충돌한 풍선을 그룹에서 제거하라는 의미이다.
- 제거 후에는 풍선을 더 만들어낸다.
- 이 코드를 반복문 안에 넣어준다.

### 🚀**해보기 3: 충돌 감지 및 제거**
<div style="background-color:rgba(247, 239, 246, 1); padding: 10px; border-radius: 5px;">
<mark style="background-color: rgba(247, 239, 246, 1);">
Ex_ch7_source.ipynb에 충돌 감지 및 제거 코드를 추가하세요.
</div>

---

이번에는 추가적으로 풍선 괴물을 게임에 넣어보고 승리와 패배 조건, 게임 레벨화를 해보기로 한다. 
- 풍선괴물 이미지 다운로드
    - https://haesunbyun.github.io/common/images/ghost.png

### 🚀**해보기 4: 목표물 추가 및 이동**

<div style="background-color:rgba(247, 239, 246, 1); padding: 10px; border-radius: 5px;">
<mark style="background-color: rgba(247, 239, 246, 1);">
Ex_ch7_source.ipynb에 Enemy_ghost 클래스 설계하고 동작하도록 코드를 넣어주세요.<br>
<p>
Enemy_ghost 클래스 설계는 다음과 같다.<br>
</mark></div>

1. 속성 (Attributes)
    - 이미지 (self.image): 고스트(유령) 이미지를 불러와서 크기 조절 (pygame.transform.scale)
        - 처음 위치는 화면 하단 근처 (예: y=550)
    - 이동 방향과 속도 (self.velocity)
        - 이동 방향: 좌우 방향 양수면 오른쪽, 음수면 왼쪽
        - 이동 속도: random.uniform()으로 속도를 무작위로 설정
    - 충돌 마스크 (self.mask)
        - 이미지 모양대로 충돌 판정이 가능하도록 pygame.mask.from_surface() 사용    

2. 동작 (Methods)

    - `__init__`() : 생성자
        - 이미지 불러오기, 크기 조절
        - 초기 위치 설정 (화면 하단에 랜덤한 x 좌표)
        - 초기 속도 무작위 설정
        - 충돌 마스크 생성

    - update() : 매 프레임마다 실행되는 동작
        - 좌우로만 바닥에서 이동
        - 벽에 닿으면 방향 반전
        - 오른쪽 끝 도달 → self.velocity를 음수로 전환 (왼쪽으로 이동)
        - 왼쪽 끝 도달 → self.velocity를 양수로 전환 (오른쪽으로 이동)

---

### ✅ 점수 기록 및 업데이트

- 플레이어의 점수를 추적하는 것은 많은 게임에서 필수적이다.
- 우리는 이미 이전 수업에서 점수를 넣는 방법을 배웠다.
- 이 게임에도 점수를 넣어 나타내보자.

In [ ]:
#점수 초기화 및 폰트 설정
score = 0
pygame.font.init()
font = pygame.font.Font(None, 36)
points = 1

#충돌할 때 점수 업데이트
score += points

def display_score(screen, score):
    score_text = font.render(f'Score: {score}', True, (0, 0, 0))
    screen.blit(score_text, (10, 10))  

### 🚀**해보기 5: 점수 시스템**
<div style="background-color:rgba(247, 239, 246, 1); padding: 10px; border-radius: 5px;">
<mark style="background-color: rgba(247, 239, 246, 1);">
Ex_ch7_source.ipynb에 점수 시스템 코드를 추가하세요.
</mark></div>

---

## <div style="background-color:rgba(208, 205, 208, 1); padding: 10px; border-radius: 5px;"><mark style="background-color: rgba(208, 205, 208, 1);">**레벨 생성**</mark></div>
- 레벨은 새로운 도전 과제, 적, 목표를 도입할 수 있다.

### ✅ 여러 레벨 디자인

- 레벨 데이터를 구조화된 형식으로 저장한다. 
- 예를 들어, 딕셔너리 리스트를 사용하여 각 딕셔너리가 하나의 레벨을 나타내고, 해당 레벨의 레이아웃, 적, 기타 요소들에 대한 정보를 포함시킬 수 있다.

In [ ]:
# 레벨과 목표 점수 설정
level = 1
max_level = 3
level_goal = {1: 5, 2: 10, 3: 15}

# 레벨업 메시지 관련
level_up_message = ""
message_timer = 0
MESSAGE_DURATION = 2000  # 2초


### ✅ 레벨 간 전환
- 레벨 간 전환은 플레이어가 현재 레벨을 완료했을 때 이를 감지하고, 다음 레벨을 로드하는 과정이다.
- 이를 위해, 플레이어가 목표에 도달했거나 레벨을 완료하는 다른 조건을 충족했는지 확인한다.
- 여기서는 간단하게 각 단계별 score 점수로만 레벨업을 해보도록 하자.
- 또한 레벨업을 했을 때 풍선유령의 개수를 늘려보도록 하자.

In [ ]:
# ghost 생성 함수
def spawn_ghosts(num):
    ghost_group.empty()  # 기존 유령 모두 제거    
    for _ in range(num):
        ghost_group.add(Enemy_ghost())


# 레벨업 조건 체크
if level <= max_level and score >= level_goal[level]:
    level_up_message = f"Level {level} Clear"
    level += 1    
    message_timer = pygame.time.get_ticks()
    spawn_ghosts(level)  # 레벨만큼 유령 생성

# 레벨업 메시지 표시 (2초간)
if level_up_message and pygame.time.get_ticks() - message_timer < MESSAGE_DURATION:
    msg_text = font.render(level_up_message, True, (255, 0, 0))
    screen.blit(msg_text, (350, 300))
else:
    level_up_message = ""               

### ✅ 승리 및 패배 조건

- 플레이어가 게임에서 승리하거나 패배하는 조건을 정의한다. 예를 들어, 플레이어는 특정 점수에 도달하면 승리하거나 적과 충돌하면 패배할 수 있다.

In [ ]:
# 충돌하면 끝나는 게임이라면 
import tkinter as tk
import tkinter.messagebox  # 메시지창을 위한 모듈 추가

if pygame.sprite.spritecollideany(player,  ghost_group):
    tkinter.messagebox.showinfo("Game Over", "'게임이 끝났습니다!'")
    running = False

In [ ]:
finish_score = 15
if score >= finish_score:
    tkinter.messagebox.showinfo("Win", "You win")
    running = False  
    

### 🚀**해보기 6: 승리 및 패배 조건**
<div style="background-color:rgba(247, 239, 246, 1); padding: 10px; border-radius: 5px;">
<mark style="background-color: rgba(247, 239, 246, 1);">
Ex_ch7_source.ipynb에 승리 및 패배 조건 코드를 추가하세요.
</mark></div>

## <div style="background-color:rgba(208, 205, 208, 1); padding: 10px; border-radius: 5px;"><mark style="background-color: rgba(208, 205, 208, 1);">**마무리**</mark></div>
오늘 수업에서는 Pygame을 활용하여 사운드와 배경음악을 적용하는 방법을 실습해 보았다.
또한 게임의 흐름에 맞는 논리 구조와 규칙을 설계하며 전체적인 진행 방식을 구현했다.
난이도 조절을 위한 레벨 구성과 장면 전환 기능을 통해 게임의 완성도를 한 단계 높였다.
이 과정에서 게임의 몰입도를 높이는 핵심 요소들을 직접 구현하며 프로그래밍 능력을 향상시킬 수 있었다.
앞으로는 이번에 익힌 기초 기술을 바탕으로 다양한 장르의 게임 제작에도 응용할 수 있을 것이다.

---
<font color='Blue' size="4">
F37.206 컴퓨팅 탐색: 실생활에서 활용하기(Exploring Computing: Applications in Everyday Life)</font>

---
서울대학교 학부대학 강의교수 변해선